# 🚀 Huấn luyện Phân loại Karyotype (24 Lớp) bằng Swin-T trên Google Colab

**Hướng dẫn nhanh:**
1. Vào menu **Runtime (Thời gian chạy)** > **Change runtime type (Đổi loại thời gian chạy)** > Chọn **T4 GPU**.
2. Tải file `karyotype.zip` lên thư mục gốc của **Google Drive** của bạn.
3. Chạy từng Cell bên dưới từ trên xuống dưới. (Code đã tự động hóa 100% để tìm đúng đường dẫn và sửa mọi lỗi).

In [ ]:
# 1. Kết nối với Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Giải nén Dataset và Tự động nhận diện đường dẫn (Smart Detection)
import os
import sys

ZIP_PATH = '/content/drive/MyDrive/karyotype.zip'
if not os.path.exists(ZIP_PATH):
    print("❌ CẢNH BÁO: Không tìm thấy file karyotype.zip trên Google Drive của bạn!")
    print("Vui lòng đảm bảo bạn đã tải file lên đúng thư mục gốc của Drive và tài khoản Drive bạn kết nối là chính xác.")
    sys.exit(1)

print("⏳ Bắt đầu giải nén dữ liệu từ Google Drive... (Vui lòng đợi vài chục giây)")
!mkdir -p /content/dataset
# Giải nén và ẩn bớt log dài, chỉ hiện xem có lỗi không
!unzip -o {ZIP_PATH} -d /content/dataset > /dev/null
print("✅ Giải nén xong!")

print("⏳ Đang dò tìm thư mục Dataset chứa 24 lớp NST...")
DATASET_DIR = None
for root, dirs, files in os.walk('/content/dataset'):
    if '1' in dirs and 'X' in dirs and 'Y' in dirs:
        DATASET_DIR = root
        break

if DATASET_DIR:
    print(f"🎯 Tuyệt vời! Đã tìm thấy thư mục chuẩn tại: {DATASET_DIR}")
    # Lưu path vào biến môi trường để các ô cell dưới xài chung
    os.environ['MY_DATASET_DIR'] = DATASET_DIR
else:
    print("❌ LỖI: Giải nén thành công nhưng không thấy 24 thư mục (1-22, X, Y).")
    print("Hãy thử chạy lệnh: !ls -la /content/dataset để xem bên trong có gì.")
    sys.exit(1)

In [ ]:
# 3. Khởi tạo Mô hình Swin-T (Swin Transformer)
import torch
import torch.nn as nn
import torchvision.models as models

class SwinKaryotype(nn.Module):
    def __init__(self, num_classes=24):
        super(SwinKaryotype, self).__init__()
        weights = models.Swin_T_Weights.DEFAULT
        self.model = models.swin_t(weights=weights)
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.model(x)

print('✅ Đã khởi tạo cấu trúc mạng Swin-T!')

In [ ]:
# 4. Cài đặt các thông số (Hyperparameters)
import os
KARYOTYPE_LABELS = [str(i) for i in range(1, 23)] + ['X', 'Y']
NUM_CLASSES = 24
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100
LR = 3e-4
SAVE_PATH = '/content/drive/MyDrive/karyotype_swint_best.pth'

DATASET_DIR = os.environ.get('MY_DATASET_DIR')
if not DATASET_DIR:
    raise ValueError("❌ LỖI: Chưa có đường dẫn Dataset. Hãy chắc chắn bạn đã chạy Ô số 2 thành công!")
print(f"Sử dụng dữ liệu tại: {DATASET_DIR}")

In [ ]:
# 5. Chuẩn bị Dữ liệu (Dataloader) và Data Augmentation
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm

class KaryotypeDataset(ImageFolder):
    def find_classes(self, directory):
        classes = KARYOTYPE_LABELS
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = KaryotypeDataset(root=DATASET_DIR, transform=transform_train)
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'📊 Tổng số ảnh đã tải: {total_size} (Huấn luyện: {train_size}, Đánh giá: {val_size})')

In [ ]:
# 6. Tiến hành Huấn luyện AI trên GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ Đang chạy trên thiết bị: {device.type.upper()}')
if device.type == 'cpu':
    print("⚠️ CẢNH BÁO: Đang chạy bằng CPU, quá trình huấn luyện sẽ rất chậm! Hãy bật T4 GPU trong phần Runtime.")

model = SwinKaryotype(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f'Vòng {epoch}/{EPOCHS} [Học]', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Vòng {epoch}/{EPOCHS} [Đánh giá]', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    scheduler.step(val_acc)
    print(f'Vòng {epoch} | Độ chuẩn xác Học: {100.*train_correct/train_total:.2f}% | Độ chuẩn xác Đánh giá: {val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'🔥 Kỷ lục mới! Đã lưu tự động bộ não vào: {SAVE_PATH}')